# Part 9: Recommendation Systems

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Building Recommendation Engines**

---
## 9.1 Collaborative Filtering

**Concept:** Recommend based on what similar users liked

**Types:**
1. **User-based:** Find similar users, recommend their items
2. **Item-based:** Find similar items, recommend to user

**Pros:** No domain knowledge needed, discovers unexpected patterns

**Cons:** Cold start problem, sparsity issues

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Create sample movie ratings dataset
np.random.seed(42)
users = ['User1', 'User2', 'User3', 'User4', 'User5']
movies = ['Movie1', 'Movie2', 'Movie3', 'Movie4', 'Movie5', 'Movie6']

# User-Item ratings matrix (0 = not rated)
ratings = pd.DataFrame([
    [5, 3, 0, 1, 0, 2],
    [4, 0, 0, 1, 2, 0],
    [1, 1, 0, 5, 4, 0],
    [1, 0, 0, 4, 0, 5],
    [0, 1, 5, 4, 0, 0]
], columns=movies, index=users)

print("User-Item Ratings Matrix:")
print(ratings)
print(f"\nSparsity: {(ratings == 0).sum().sum() / ratings.size * 100:.1f}%")

### User-Based Collaborative Filtering

In [ ]:
def user_based_cf(ratings_matrix, user, n_similar=3):
    """User-based collaborative filtering"""
    
    # Replace 0 with NaN for similarity calculation
    ratings_filled = ratings_matrix.replace(0, np.nan)
    
    # Calculate user similarity (cosine)
    user_similarity = cosine_similarity(ratings_matrix.fillna(0))
    user_sim_df = pd.DataFrame(user_similarity, 
                               index=ratings_matrix.index, 
                               columns=ratings_matrix.index)
    
    # Find similar users
    similar_users = user_sim_df[user].sort_values(ascending=False)[1:n_similar+1]
    print(f"Most similar users to {user}:")
    print(similar_users)
    
    # Get movies not rated by target user
    user_unrated = ratings_matrix.loc[user] == 0
    
    # Predict ratings using weighted average
    predictions = {}
    for movie in ratings_matrix.columns[user_unrated]:
        # Get ratings from similar users for this movie
        similar_ratings = ratings_matrix.loc[similar_users.index, movie]
        similar_ratings = similar_ratings[similar_ratings > 0]
        
        if len(similar_ratings) > 0:
            # Weighted average by similarity
            weights = similar_users[similar_ratings.index]
            weighted_rating = (similar_ratings * weights).sum() / weights.sum()
            predictions[movie] = weighted_rating
    
    # Sort recommendations
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations

# Get recommendations
user = 'User1'
recommendations = user_based_cf(ratings, user)
print(f"\nRecommendations for {user}:")
print(recommendations)

### Item-Based Collaborative Filtering

In [ ]:
def item_based_cf(ratings_matrix, user, n_similar=3):
    """Item-based collaborative filtering"""
    
    # Calculate item similarity
    item_similarity = cosine_similarity(ratings_matrix.T.fillna(0))
    item_sim_df = pd.DataFrame(item_similarity,
                               index=ratings_matrix.columns,
                               columns=ratings_matrix.columns)
    
    # Get user's rated items
    user_ratings = ratings_matrix.loc[user]
    rated_items = user_ratings[user_ratings > 0]
    
    # Get unrated items
    unrated_items = user_ratings[user_ratings == 0].index
    
    # Predict ratings
    predictions = {}
    for item in unrated_items:
        # Find similar items that user has rated
        similar_items = item_sim_df[item][rated_items.index]
        
        if len(similar_items) > 0:
            # Weighted average
            weighted_rating = (rated_items * similar_items).sum() / similar_items.sum()
            predictions[item] = weighted_rating
    
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations

# Get recommendations
recommendations_item = item_based_cf(ratings, user)
print(f"Item-based recommendations for {user}:")
print(recommendations_item)

### Matrix Factorization (SVD)

**Concept:** Decompose user-item matrix into latent factors

**Advantages:**
- Handles sparsity better
- Captures latent features
- Scalable

In [ ]:
# Install: pip install scikit-surprise
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

# Prepare data for Surprise library
# Convert to long format
data_list = []
for user in ratings.index:
    for movie in ratings.columns:
        rating = ratings.loc[user, movie]
        if rating > 0:  # only rated items
            data_list.append([user, movie, rating])

df = pd.DataFrame(data_list, columns=['user', 'item', 'rating'])

# Create Surprise dataset
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user', 'item', 'rating']], reader)

# Split data
trainset, testset = train_test_split(data, test_size=0.25)

# SVD model
svd = SVD(
    n_factors=10,        # number of latent factors
    n_epochs=20,         # training epochs
    lr_all=0.005,        # learning rate
    reg_all=0.02         # regularization
)

# Train
svd.fit(trainset)

# Predict
predictions = svd.test(testset)
accuracy.rmse(predictions)

# Get recommendations for a user
def get_svd_recommendations(model, user_id, items, ratings_df, n=5):
    # Get unrated items
    rated_items = ratings_df.loc[user_id][ratings_df.loc[user_id] > 0].index
    unrated_items = [item for item in items if item not in rated_items]
    
    # Predict ratings
    predictions = [(item, model.predict(user_id, item).est) 
                  for item in unrated_items]
    
    # Sort and return top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]

recommendations_svd = get_svd_recommendations(svd, 'User1', movies, ratings)
print(f"\nSVD Recommendations for User1:")
for item, score in recommendations_svd:
    print(f"{item}: {score:.2f}")

---
## 9.2 Content-Based Filtering

**Concept:** Recommend items similar to what user liked (based on features)

**Pros:** No cold start for items, explainable

**Cons:** Limited discovery, needs feature engineering

In [ ]:
# Movie features dataset
movie_features = pd.DataFrame({
    'movie': ['Movie1', 'Movie2', 'Movie3', 'Movie4', 'Movie5', 'Movie6'],
    'genre': ['Action', 'Comedy', 'Action', 'Drama', 'Drama', 'Action'],
    'year': [2020, 2019, 2021, 2018, 2020, 2019],
    'director': ['A', 'B', 'A', 'C', 'C', 'A'],
    'rating': [7.5, 6.8, 8.2, 7.0, 7.8, 7.9]
})

print("Movie Features:")
print(movie_features)

# One-hot encode categorical features
features_encoded = pd.get_dummies(movie_features, columns=['genre', 'director'])
features_encoded = features_encoded.drop('movie', axis=1)

# Normalize numerical features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_encoded[['year', 'rating']] = scaler.fit_transform(
    features_encoded[['year', 'rating']])

print("\nEncoded Features:")
print(features_encoded)

# Calculate item similarity
item_similarity = cosine_similarity(features_encoded)
item_sim_df = pd.DataFrame(item_similarity,
                           index=movie_features['movie'],
                           columns=movie_features['movie'])

print("\nItem Similarity Matrix:")
print(item_sim_df)

In [ ]:
def content_based_recommendations(user, ratings_df, item_sim_df, n=3):
    """Content-based recommendations"""
    
    # Get user's rated items
    user_ratings = ratings_df.loc[user]
    liked_items = user_ratings[user_ratings >= 4].index  # items rated >= 4
    
    if len(liked_items) == 0:
        return pd.Series()
    
    # Get unrated items
    unrated_items = user_ratings[user_ratings == 0].index
    
    # Calculate scores for unrated items
    scores = {}
    for item in unrated_items:
        # Average similarity to liked items
        similarity_scores = item_sim_df.loc[liked_items, item]
        scores[item] = similarity_scores.mean()
    
    # Sort and return top N
    recommendations = pd.Series(scores).sort_values(ascending=False)[:n]
    return recommendations

# Get recommendations
content_recs = content_based_recommendations('User1', ratings, item_sim_df)
print(f"Content-based recommendations for User1:")
print(content_recs)

---
## 9.3 Hybrid Approaches

**Concept:** Combine collaborative and content-based methods

**Strategies:**
1. **Weighted:** Combine scores with weights
2. **Switching:** Choose method based on context
3. **Mixed:** Present results from both
4. **Feature combination:** Use collaborative features in content-based
5. **Meta-level:** Use one method's output as input to another

In [ ]:
def hybrid_recommendations(user, ratings_df, item_sim_df, 
                          cf_weight=0.5, cb_weight=0.5, n=3):
    """Hybrid recommendation system"""
    
    # Get collaborative filtering scores
    cf_scores = item_based_cf(ratings_df, user)
    
    # Get content-based scores
    cb_scores = content_based_recommendations(user, ratings_df, item_sim_df, n=10)
    
    # Normalize scores to [0, 1]
    if len(cf_scores) > 0:
        cf_scores = (cf_scores - cf_scores.min()) / (cf_scores.max() - cf_scores.min())
    if len(cb_scores) > 0:
        cb_scores = (cb_scores - cb_scores.min()) / (cb_scores.max() - cb_scores.min())
    
    # Combine scores
    all_items = set(cf_scores.index) | set(cb_scores.index)
    hybrid_scores = {}
    
    for item in all_items:
        cf_score = cf_scores.get(item, 0)
        cb_score = cb_scores.get(item, 0)
        hybrid_scores[item] = cf_weight * cf_score + cb_weight * cb_score
    
    # Sort and return top N
    recommendations = pd.Series(hybrid_scores).sort_values(ascending=False)[:n]
    return recommendations

# Get hybrid recommendations
hybrid_recs = hybrid_recommendations('User1', ratings, item_sim_df, 
                                    cf_weight=0.6, cb_weight=0.4)
print(f"Hybrid recommendations for User1:")
print(hybrid_recs)

### Deep Learning for Recommendations

**Neural Collaborative Filtering (NCF)**

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import LabelEncoder

# Prepare data
data_list = []
for user in ratings.index:
    for movie in ratings.columns:
        rating = ratings.loc[user, movie]
        if rating > 0:
            data_list.append([user, movie, rating])

df = pd.DataFrame(data_list, columns=['user', 'movie', 'rating'])

# Encode users and items
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user'])
df['movie_id'] = movie_encoder.fit_transform(df['movie'])

n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()

# Build NCF model
embedding_size = 8

# User embedding
user_input = layers.Input(shape=(1,), name='user_input')
user_embedding = layers.Embedding(n_users, embedding_size, name='user_embedding')(user_input)
user_vec = layers.Flatten(name='user_flatten')(user_embedding)

# Movie embedding
movie_input = layers.Input(shape=(1,), name='movie_input')
movie_embedding = layers.Embedding(n_movies, embedding_size, name='movie_embedding')(movie_input)
movie_vec = layers.Flatten(name='movie_flatten')(movie_embedding)

# Concatenate and predict
concat = layers.Concatenate()([user_vec, movie_vec])
dense1 = layers.Dense(64, activation='relu')(concat)
dense2 = layers.Dense(32, activation='relu')(dense1)
output = layers.Dense(1, activation='linear')(dense2)

model = Model(inputs=[user_input, movie_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print(model.summary())

# Train
X_user = df['user_id'].values
X_movie = df['movie_id'].values
y = df['rating'].values

history = model.fit(
    [X_user, X_movie], y,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=0
)

print(f"\nFinal Loss: {history.history['loss'][-1]:.4f}")

---
### Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_recommendations(predictions, actuals):
    """Evaluate recommendation quality"""
    
    # Rating prediction metrics
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    return rmse, mae

def precision_recall_at_k(recommendations, relevant_items, k=5):
    """Calculate Precision@K and Recall@K"""
    
    top_k = recommendations[:k]
    
    # Precision@K
    relevant_in_top_k = len(set(top_k) & set(relevant_items))
    precision = relevant_in_top_k / k if k > 0 else 0
    
    # Recall@K
    recall = relevant_in_top_k / len(relevant_items) if len(relevant_items) > 0 else 0
    
    return precision, recall

# Example
recommended = ['Movie3', 'Movie5', 'Movie6', 'Movie1', 'Movie2']
relevant = ['Movie3', 'Movie5', 'Movie4']

precision, recall = precision_recall_at_k(recommended, relevant, k=3)
print(f"\nPrecision@3: {precision:.2f}")
print(f"Recall@3: {recall:.2f}")

---
### Quick Reference Guide

**Method Selection:**

| Method | Cold Start | Sparsity | Scalability | Discovery |
|--------|-----------|----------|-------------|----------|
| User-based CF | Poor | Poor | Medium | Good |
| Item-based CF | Poor | Medium | Good | Good |
| Matrix Factorization | Poor | Good | Good | Good |
| Content-based | Good (items) | Good | Good | Poor |
| Hybrid | Good | Good | Medium | Good |
| Deep Learning | Medium | Good | Medium | Good |

**Best Practices:**
- Start with item-based CF (better than user-based)
- Use matrix factorization for sparse data
- Combine methods in hybrid approach
- Consider context (time, location, device)
- Implement A/B testing
- Monitor diversity and novelty
- Handle cold start with content-based or popularity
- Use implicit feedback when available

**Common Challenges:**
1. **Cold Start:** New users/items have no history
2. **Sparsity:** Most user-item pairs unrated
3. **Scalability:** Large user/item spaces
4. **Diversity:** Avoid filter bubble
5. **Popularity Bias:** Don't just recommend popular items

---
[← Back to Index](Index.ipynb)